In [2]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch import nn
import tensorflow as tf

In [3]:
Npontos = 60 #Tempo de dados 60
features = 6

In [4]:
dataset = pd.read_csv('completo.csv')

In [5]:
dataset.tail()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
3368537,1650,17,2426184371437252,2.137723,-7.211485,8.065467,-2.856065,2.057101,0.853933
3368538,1650,17,2426184421424592,1.486500,-13.302334,5.355231,-2.911458,1.817417,1.281104
3368539,1650,17,2426184471411932,4.017171,-13.146711,5.118205,-3.015854,1.550036,2.057682
3368540,1650,17,2426184521399272,4.412214,-15.318253,3.370437,-3.218254,1.595842,3.018550
3368541,1650,17,2426184571385262,4.426579,-13.012636,2.575563,-2.455525,0.866136,3.354109


In [6]:
parc_data_train = 0.8
andando = dataset.loc[dataset['atividade'] == 0]
split = int(len(andando)*parc_data_train)
train_andando = andando[:split]
teste_andando = andando[split:]


In [7]:
sentado = dataset.loc[dataset['atividade'] == 3]
split = int(len(sentado)*parc_data_train)
train_sentado = sentado[:split]
teste_sentado = sentado[split:]


In [8]:
escreve = dataset.loc[dataset['atividade'] == 15]
split = int(len(escreve)*parc_data_train)
train_escreve = escreve[:split]
teste_escreve = escreve[split:]

In [9]:
escada = dataset.loc[dataset['atividade'] == 2]
split = int(len(escada)*parc_data_train)
train_escada = escada[:split]
teste_escada = escada[split:]
treino = pd.concat([train_andando, train_sentado, train_escreve, train_escada])
teste = pd.concat([teste_andando, teste_sentado, teste_escreve, teste_escada])
treino = treino.replace({2: 1, 3: 2, 15:3})
teste = teste.replace({2: 1, 3: 2, 15:3})

In [10]:
label_teste = teste['atividade']
label_teste = label_teste.values
teste = teste[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']].values

In [11]:
label = treino['atividade']
treino = treino[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']].values
label = label.values

In [13]:
print(treino)

[[ 4.972757   -0.15831658  6.6967316   0.3149441  -1.0222765  -0.3099616 ]
 [ 3.25372    -0.19183542  6.107758    0.3873821  -0.61854124 -0.0489718 ]
 [ 2.801216   -0.15592238  5.997625    0.07099854 -0.20947972 -0.1959783 ]
 ...
 [-2.6225464  -4.4188113   4.3421707   0.95660746 -0.7755125   0.33449304]
 [-2.8740237  -3.9493873   4.605623    0.6018744  -0.56459016  0.20879184]
 [-2.823728   -3.796106    4.8259645   0.25992453 -0.38775626  0.10546118]]


In [20]:
print(label)

[0 0 0 ... 1 1 1]


In [14]:
def cria_IN_OUT(data, janela, features):
    OUT_indices = np.arange(janela, len(data), janela)
    OUT = data[OUT_indices]
    IN = data[range(janela*len(OUT))]
    IN = np.reshape(IN, (len(OUT), janela, features))
    IN_final = IN[:,:,:]
    return IN_final
trainX = cria_IN_OUT(treino, Npontos, 6)
trainY = cria_IN_OUT(label, Npontos, 1)
testX = cria_IN_OUT(teste, Npontos, 6)
testY = cria_IN_OUT(label_teste, Npontos, 1)


In [15]:
media = np.mean(testX, axis=0)
desvio_padrao = np.std(testX)
testXn = (testX - media) / desvio_padrao
media = np.mean(trainX, axis=0)
desvio_padrao = np.std(trainX)
trainXn = (trainX - media) / desvio_padrao

In [16]:
testY = testY.astype(int)
testY.dtype
trainY = trainY.astype(int)

In [17]:
#teste de Label
for i in range(0,257,1):
    teste = False
    for x in range(0,58,1):
        if teste == False:
            teste = trainY[i][x][0]
        else:
            if teste != trainY[i][x][0]:
                print('errado')
print(trainXn[0][1])
print(trainX[0][1])

[-0.01852131  0.86642802  0.8763117   0.08673388 -0.13810259 -0.01077467]
[ 3.25372    -0.19183542  6.107758    0.3873821  -0.61854124 -0.0489718 ]


In [18]:
from keras.regularizers import l2

In [ ]:
def cria_LSTM(hidden_units, dense_units, input_shape, activation):
  model = tf.keras.Sequential()
  model.add(tf.keras.layers.LSTM(hidden_units, input_shape=input_shape,activation=activation[0], return_sequences=True))
  tf.keras.layers.BatchNormalization()
  #model.add(tf.keras.layers.Dense(units=64, activation=activation[0]))
  model.add(tf.keras.layers.Dropout(rate=0.5))
  #model.add(tf.keras.layers.Dense(units=32, activation=activation[0]))
  model.add(tf.keras.layers.Dense(units=dense_units, activation=activation[1]))
  model.compile(loss='sparse_categorical_crossentropy', optimizer ='adam', metrics=['accuracy'])
  return model

In [ ]:

model = cria_LSTM(hidden_units=120, dense_units=4, input_shape=(Npontos, features), activation=['relu', 'softmax'])
resultado = []
for i in range(20):
    model.reset_states()
    history = model.fit(trainXn, trainY, epochs = 50, batch_size = 30, verbose = 1,  validation_data=(testXn, testY))
    resultado.append(history)


Epoch 1/50
336/336 [==============================] - 19s 50ms/step - loss: 0.9703 - accuracy: 0.5988 - val_loss: 0.8323 - val_accuracy: 0.6197
Epoch 2/50
336/336 [==============================] - 20s 58ms/step - loss: 0.7752 - accuracy: 0.6298 - val_loss: 0.7915 - val_accuracy: 0.6295
Epoch 3/50
336/336 [==============================] - 16s 49ms/step - loss: 2.8761 - accuracy: 0.6319 - val_loss: 0.7598 - val_accuracy: 0.6352
Epoch 4/50
336/336 [==============================] - 16s 49ms/step - loss: 0.7241 - accuracy: 0.6398 - val_loss: 0.7403 - val_accuracy: 0.6418
Epoch 5/50
336/336 [==============================] - 17s 50ms/step - loss: 0.6801 - accuracy: 0.6627 - val_loss: 0.7305 - val_accuracy: 0.6446
Epoch 6/50
336/336 [==============================] - 19s 56ms/step - loss: 0.6602 - accuracy: 0.6707 - val_loss: 0.7120 - val_accuracy: 0.6734
Epoch 7/50
234/336 [===================>..........] - ETA: 4s - loss: 0.6491 - accuracy: 0.6792

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# Seu código para criar o modelo e treinar

# Criar subplots
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 8))

# Plotar a perda (loss) nos subplots[0]
axes[0].set_title('Perda ao longo das épocas')
for i in range(20):
    axes[0].plot(resultado[i].history['loss'], label=f'Treino {i + 1}')

# Plotar a acurácia nos subplots[1]
axes[1].set_title('Acurácia ao longo das épocas')
for i in range(20):
    axes[1].plot(resultado[i].history['accuracy'], label=f'Treino {i + 1}')

# Adicionar legendas
axes[0].legend()
axes[1].legend()

# Ajustar layout
plt.tight_layout()

# Exibir os gráficos
plt.show()

In [ ]:
plt.figure(figsize= (7,7))

plt.subplot(2,1,1)
plt.title("loss")
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'], 'k')
plt.legend(['treino', 'teste'])

plt.subplot(2,1,2)
plt.title("acurácia")
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'], 'k')
plt.legend(['treino', 'teste'])
plt.xlabel('epoch')

In [ ]:
accuracy = model.evaluate(testX, testY)
print('Acurácia: %.2f%%' % (accuracy[1]*100))
print('Loss: %.2f' % (accuracy[0]))


In [6]:
import pandas as pd
import numpy as np

# Generate random data for each feature
data = {
    "Feature1": np.random.randint(0, 100, 20),  # Integer values between 0 and 99
    "Feature2": np.random.randn(20),  # Normally distributed random values
    "Feature3": np.random.choice(["A", "B", "C"], size=20, p=[0.4, 0.3, 0.3])  # Categorical data with specific probabilities
}

# Create the DataFrame
df = pd.DataFrame(data)

print(df)

    Feature1  Feature2 Feature3
0         53 -0.102877        A
1         33  0.477179        A
2         16  0.395167        C
3         38  1.447296        A
4          7 -1.181704        A
5         71  0.266738        C
6          1  0.278470        C
7         19 -0.243937        A
8         22 -1.310503        C
9         57  0.041456        A
10        33 -0.329634        C
11         2 -2.300349        A
12        92 -0.044132        A
13         7 -1.096055        A
14        79 -0.531762        A
15        34  0.585559        B
16        28  0.154892        B
17        47  1.226666        C
18         9 -0.832085        A
19        18  0.441662        A


In [8]:
import pandas as pd

# Sample DataFrame with 3 features and 20 rows
data = {
    "Feature1": np.random.randint(0, 100, 20),
    "Feature2": np.random.randn(20),
    "Feature3": np.random.choice(["A", "B", "C"], size=20, p=[0.4, 0.3, 0.3])
}
df = pd.DataFrame(data)

# Define split indices
split_indices = [[0, 4], [5, 9], [10, 14], [15, 19]]

# Perform and print splits
for start_index, end_index in split_indices:
    print(start_index)
    print(end_index)
    test_df = df.iloc[start_index:end_index + 1]
    train_df = df.drop(index=df.index[start_index:end_index + 1])

    print(f"\nTest data (indices {start_index}-{end_index}):")
    print(test_df)

    print(f"\nTrain data (excluding indices {start_index}-{end_index}):")
    print(train_df)

0
4

Test data (indices 0-4):
   Feature1  Feature2 Feature3
0         8  0.430818        A
1        69 -1.654119        A
2        47  2.208333        B
3         3 -2.542015        C
4        76 -0.446820        A

Train data (excluding indices 0-4):
    Feature1  Feature2 Feature3
5         53  0.419928        A
6         21 -1.372204        A
7         86 -0.185456        B
8         28  1.434828        A
9         61  0.429443        A
10        94 -1.089163        A
11        31 -0.716235        C
12        50 -0.717231        C
13        67  1.098922        B
14        28  1.553820        A
15        37  0.501665        B
16        83 -1.905034        C
17        24  0.348249        B
18        25 -0.161360        C
19        67  1.835962        C
5
9

Test data (indices 5-9):
   Feature1  Feature2 Feature3
5        53  0.419928        A
6        21 -1.372204        A
7        86 -0.185456        B
8        28  1.434828        A
9        61  0.429443        A

Train data (exclud